# Exploratory only

Source of truth = Python modules under `app/` and `scripts/`. These notebooks are retained for EDA, experiment notes, and demo walkthroughs only.

# Oriented Object Detection in Aerial Imagery (YOLO OBB)

## Objective
Build a robust object detection pipeline for aerial images with a focus on small object detection.

## Key challenges
- Small object detection (vehicles)
- High-resolution images
- Oriented bounding boxes (OBB)

## Approach
1. Baseline training
2. Problem analysis
3. Tiling strategy
4. Improved training

# Part 1 — Baseline Training with YOLO OBB

## Environment setup

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import ultralytics
print(ultralytics.__version__)

##Dataset Preparation Check

## Dataset Preparation

The original dataset follows the DOTA annotation format, which uses oriented bounding boxes defined by 4 points.

To make the dataset compatible with YOLO OBB, we perform a structured preprocessing pipeline:

### Steps

* Extract the dataset from compressed format
* Reorganize images and labels into YOLO directory structure
* Convert annotations from DOTA format to YOLO OBB format
* Normalize coordinates to the [0, 1] range

### Output format

Each label is converted into the following structure:

[class_id x1 y1 x2 y2 x3 y3 x4 y4]

Where:

* (x1, y1) … (x4, y4) represent the four corners of the oriented bounding box
* Coordinates are normalized relative to image dimensions

This step ensures full compatibility with Ultralytics YOLO OBB training pipeline.


In [ ]:
import zipfile, os

ZIP_PATH = "/content/processed.zip"
EXTRACT_PATH = "/content/data"

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset ready")

In [ ]:
DATA_ROOT = "/content/data/processed/split"

print("Root:", os.listdir(DATA_ROOT))
print("Train:", os.listdir(os.path.join(DATA_ROOT, "train")))
print("Val:", os.listdir(os.path.join(DATA_ROOT, "val")))

In [ ]:
import os
import shutil

SRC_ROOT = "/content/data/processed/split"
DST_ROOT = "/content/data/yolo_obb_raw"

for sub in [
    "images/train", "images/val",
    "labels/train", "labels/val"
]:
    os.makedirs(os.path.join(DST_ROOT, sub), exist_ok=True)

# Copier les images
for f in os.listdir(os.path.join(SRC_ROOT, "train", "images")):
    shutil.copy(
        os.path.join(SRC_ROOT, "train", "images", f),
        os.path.join(DST_ROOT, "images", "train", f)
    )

for f in os.listdir(os.path.join(SRC_ROOT, "val", "images")):
    shutil.copy(
        os.path.join(SRC_ROOT, "val", "images", f),
        os.path.join(DST_ROOT, "images", "val", f)
    )

# Copier les labels DOTA txt vers labels/train et labels/val
for f in os.listdir(os.path.join(SRC_ROOT, "train", "labelTxt")):
    shutil.copy(
        os.path.join(SRC_ROOT, "train", "labelTxt", f),
        os.path.join(DST_ROOT, "labels", "train", f)
    )

for f in os.listdir(os.path.join(SRC_ROOT, "val", "labelTxt")):
    shutil.copy(
        os.path.join(SRC_ROOT, "val", "labelTxt", f),
        os.path.join(DST_ROOT, "labels", "val", f)
    )

print("Dataset brut YOLO OBB prêt")

In [ ]:
print("Train images:", len(os.listdir("/content/data/yolo_obb_raw/images/train")))
print("Val images:", len(os.listdir("/content/data/yolo_obb_raw/images/val")))
print("Train labels:", len(os.listdir("/content/data/yolo_obb_raw/labels/train")))
print("Val labels:", len(os.listdir("/content/data/yolo_obb_raw/labels/val")))

In [ ]:
from pathlib import Path

from app.dataset_tiling import read_dota_label

CLASS_MAP = {
    "plane": 0,
    "ship": 1,
    "small-vehicle": 2,
    "large-vehicle": 3,
}

# Operational dataset conversion now lives in scripts/prepare_data.py
# and app/dataset_tiling.py. This notebook keeps only exploratory context.


In [ ]:
import random
sample_label = random.choice(os.listdir("/content/data/yolo_obb_raw/labels/train"))
print("Exemple fichier:", sample_label)

with open(os.path.join("/content/data/yolo_obb_raw/labels/train", sample_label), "r") as f:
    for _ in range(5):
        line = f.readline()
        if not line:
            break
        print(line.strip())

In [ ]:
img_set = set(f.replace(".png", "") for f in os.listdir("/content/data/yolo_obb_raw/images/train"))
lbl_set = set(f.replace(".txt", "") for f in os.listdir("/content/data/yolo_obb_raw/labels/train"))

print("Mismatch train:", len(img_set - lbl_set))

In [ ]:
img_set = set(f.replace(".png", "") for f in os.listdir("/content/data/yolo_obb_raw/images/val"))
lbl_set = set(f.replace(".txt", "") for f in os.listdir("/content/data/yolo_obb_raw/labels/val"))

print("Mismatch val:", len(img_set - lbl_set))

The dataset was successfully converted to YOLO OBB format and validated before training.

## Initial Training Configuration

## Baseline Training

We train an initial YOLO OBB model to evaluate performance.

Goal:
- Establish a baseline
- Identify weaknesses (especially small objects)

In [ ]:
yaml_text = """
path: /content/data/yolo_obb_raw
train: images/train
val: images/val

names:
  0: plane
  1: ship
  2: small-vehicle
  3: large-vehicle
"""

with open("/content/data/yolo_obb_raw/dota4_obb.yaml", "w") as f:
    f.write(yaml_text)

print("YAML créé")

In [ ]:
!cat /content/data/yolo_obb_raw/dota4_obb.yaml

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-obb.pt")

In [ ]:
results = model.train(
    data="/content/data/yolo_obb_raw/dota4_obb.yaml",
    epochs=40,
    imgsz=1280,
    batch=4,
    device=0,
    workers=2,
    patience=15,
    lr0=0.0005,

    degrees=60,
    translate=0.15,
    scale=0.5,
    fliplr=0.5,
    flipud=0.5,
    mosaic=1.0,

    project="/content/runs_obb",
    name="dota4_train"
   )

## Baseline Results Analysis

### Quantitative Results

* mAP50: **0.757**
* mAP50-95: **0.424**
* Precision: **0.886**
* Recall: **0.681**

### Class-wise Performance

* Plane: **0.978 mAP50** → excellent detection
* Ship: **0.760 mAP50** → good performance
* Large vehicle: **0.856 mAP50** → strong detection
* Small vehicle: **0.454 mAP50** → weak performance

### Observations

The model performs well on large and medium-sized objects but struggles significantly with small objects.

### Conclusion

The low performance on small vehicles highlights a scale-related limitation.
Objects occupy too few pixels in high-resolution images, making feature extraction difficult.


# Part 2 — Tiling for Small Object Detection

## Motivation for Tiling

The baseline results reveal a major limitation: poor detection of small objects such as vehicles.

### Root Causes

* Small objects occupy very few pixels in high-resolution images
* Important spatial details are lost during downsampling in the network
* Complex scenes reduce the model’s ability to focus on small targets

### Proposed Solution: Tiling

To address this issue, we apply a tiling strategy:

* Split large images into smaller patches
* Increase the relative size of objects within each tile
* Improve local feature representation
* Reduce scene complexity

This approach is expected to significantly improve detection performance for small objects.



##Imports + Tiling Parameters

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# -------------------------
# PARAMÈTRES
# -------------------------
TILE_SIZE = 1024
OVERLAP = 200
EMPTY_TILE_KEEP_RATIO = 0.10  # garder 10% des tiles sans objet

# -------------------------
# CHEMINS
# -------------------------
SRC_ROOT = "/content/data/yolo_obb_raw"
DST_ROOT = "/content/data/yolo_obb_tiled"

os.makedirs(DST_ROOT, exist_ok=True)

print("SRC_ROOT:", SRC_ROOT)
print("DST_ROOT:", DST_ROOT)

##  Tiled Dataset Generation

In [ ]:
for split in ["train", "val"]:
    os.makedirs(os.path.join(DST_ROOT, "images", split), exist_ok=True)
    os.makedirs(os.path.join(DST_ROOT, "labels", split), exist_ok=True)

print("Structure du dataset tiled créée")

In [ ]:
from pathlib import Path

from app.dataset_tiling import (
    SourceObject,
    TileWindow,
    build_tile_annotations,
    deterministic_keep_empty_tile,
    extract_padded_tile,
    generate_tile_windows,
    read_dota_label,
)

CLASS_MAP = {
    "plane": 0,
    "ship": 1,
    "small-vehicle": 2,
    "large-vehicle": 3,
}


def load_labels(label_path):
    objects = read_dota_label(Path(label_path), class_to_id=CLASS_MAP)
    return [
        (obj.class_id, [coord for point in obj.polygon for coord in point])
        for obj in objects
    ]


def generate_tiles(img, tile_size=1024, overlap=200):
    windows = generate_tile_windows(img.shape[1], img.shape[0], tile_size=tile_size, overlap=overlap)
    return [
        (extract_padded_tile(img, window, tile_size), window.x, window.y, window.width, window.height)
        for window in windows
    ]


def adjust_labels(objects, x_offset, y_offset, tile_size, img_w, img_h, tile_w, tile_h):
    tile_window = TileWindow(index=0, x=x_offset, y=y_offset, width=tile_w, height=tile_h)
    source_objects = [
        SourceObject(
            class_id=cls_id,
            class_name=str(cls_id),
            polygon=tuple((coords[i] * img_w, coords[i + 1] * img_h) for i in range(0, 8, 2)),
        )
        for cls_id, coords in objects
    ]
    return build_tile_annotations(source_objects, tile_window, tile_size)


def process_split(split, tile_size=1024, overlap=200, empty_keep_ratio=0.10):
    img_dir = os.path.join(SRC_ROOT, "images", split)
    lbl_dir = os.path.join(SRC_ROOT, "labels", split)
    out_img_dir = os.path.join(DST_ROOT, "images", split)
    out_lbl_dir = os.path.join(DST_ROOT, "labels", split)

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    saved_tiles = 0
    saved_empty_tiles = 0
    img_files = [f for f in os.listdir(img_dir) if f.endswith(".png")]

    for img_name in tqdm(img_files, desc=f"Tiling {split}"):
        img_path = os.path.join(img_dir, img_name)
        lbl_path = os.path.join(lbl_dir, img_name.replace(".png", ".txt"))

        img = cv2.imread(img_path)
        if img is None:
            continue

        source_objects = read_dota_label(Path(lbl_path), class_to_id=CLASS_MAP)
        windows = generate_tile_windows(img.shape[1], img.shape[0], tile_size, overlap)

        for window in windows:
            new_objs = build_tile_annotations(source_objects, window, tile_size)
            if not new_objs and not deterministic_keep_empty_tile(
                img_name.replace(".png", ""),
                window.index,
                empty_keep_ratio,
                42,
            ):
                continue

            tile = extract_padded_tile(img, window, tile_size)
            tile_base = f"{img_name.replace('.png', '')}_{window.index:04d}"
            tile_img_name = f"{tile_base}.png"
            tile_lbl_name = f"{tile_base}.txt"

            cv2.imwrite(os.path.join(out_img_dir, tile_img_name), tile)
            with open(os.path.join(out_lbl_dir, tile_lbl_name), "w", encoding="utf-8") as handle:
                for cls_id, coords in new_objs:
                    handle.write(str(cls_id) + " " + " ".join(f"{coord:.6f}" for coord in coords) + "\n")

            saved_tiles += 1
            if not new_objs:
                saved_empty_tiles += 1

    print(f"\nSplit: {split}")
    print(f"Tiles sauvegardees: {saved_tiles}")
    print(f"Tiles vides conservees: {saved_empty_tiles}")


In [ ]:
from pathlib import Path

from app.dataset_tiling import (
    SourceObject,
    build_tile_annotations,
    deterministic_keep_empty_tile,
    extract_padded_tile,
    generate_tile_windows,
    read_dota_label,
)

CLASS_MAP = {
    "plane": 0,
    "ship": 1,
    "small-vehicle": 2,
    "large-vehicle": 3,
}


def load_labels(label_path):
    objects = read_dota_label(Path(label_path), class_to_id=CLASS_MAP)
    return [
        (obj.class_id, [coord for point in obj.polygon for coord in point])
        for obj in objects
    ]


def generate_tiles(img, tile_size=1024, overlap=200):
    windows = generate_tile_windows(img.shape[1], img.shape[0], tile_size=tile_size, overlap=overlap)
    return [
        (extract_padded_tile(img, window, tile_size), window.x, window.y, window.width, window.height)
        for window in windows
    ]


def adjust_labels(objects, x_offset, y_offset, tile_size, img_w, img_h, tile_w, tile_h):
    window = SourceObject, None
    tile_window = __import__("app.dataset_tiling", fromlist=["TileWindow"]).TileWindow(
        index=0,
        x=x_offset,
        y=y_offset,
        width=tile_w,
        height=tile_h,
    )
    source_objects = [
        SourceObject(
            class_id=cls_id,
            class_name=str(cls_id),
            polygon=tuple((coords[i] * img_w, coords[i + 1] * img_h) for i in range(0, 8, 2)),
        )
        for cls_id, coords in objects
    ]
    return build_tile_annotations(source_objects, tile_window, tile_size)


def process_split(split, tile_size=1024, overlap=200, empty_keep_ratio=0.10):
    img_dir = os.path.join(SRC_ROOT, "images", split)
    lbl_dir = os.path.join(SRC_ROOT, "labels", split)
    out_img_dir = os.path.join(DST_ROOT, "images", split)
    out_lbl_dir = os.path.join(DST_ROOT, "labels", split)

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    saved_tiles = 0
    saved_empty_tiles = 0
    img_files = [f for f in os.listdir(img_dir) if f.endswith(".png")]

    for img_name in tqdm(img_files, desc=f"Tiling {split}"):
        img_path = os.path.join(img_dir, img_name)
        lbl_path = os.path.join(lbl_dir, img_name.replace(".png", ".txt"))

        img = cv2.imread(img_path)
        if img is None:
            continue

        source_objects = read_dota_label(Path(lbl_path), class_to_id=CLASS_MAP)
        windows = generate_tile_windows(img.shape[1], img.shape[0], tile_size, overlap)

        for window in windows:
            new_objs = build_tile_annotations(source_objects, window, tile_size)
            if not new_objs and not deterministic_keep_empty_tile(
                img_name.replace(".png", ""),
                window.index,
                empty_keep_ratio,
                42,
            ):
                continue

            tile = extract_padded_tile(img, window, tile_size)
            tile_base = f"{img_name.replace('.png', '')}_{window.index:04d}"
            tile_img_name = f"{tile_base}.png"
            tile_lbl_name = f"{tile_base}.txt"

            cv2.imwrite(os.path.join(out_img_dir, tile_img_name), tile)
            with open(os.path.join(out_lbl_dir, tile_lbl_name), "w", encoding="utf-8") as handle:
                for cls_id, coords in new_objs:
                    handle.write(str(cls_id) + " " + " ".join(f"{coord:.6f}" for coord in coords) + "\n")

            saved_tiles += 1
            if not new_objs:
                saved_empty_tiles += 1

    print(f"\nSplit: {split}")
    print(f"Tiles sauvegardees: {saved_tiles}")
    print(f"Tiles vides conservees: {saved_empty_tiles}")


In [ ]:
from pathlib import Path

from app.dataset_tiling import (
    SourceObject,
    build_tile_annotations,
    deterministic_keep_empty_tile,
    extract_padded_tile,
    generate_tile_windows,
    read_dota_label,
)

CLASS_MAP = {
    "plane": 0,
    "ship": 1,
    "small-vehicle": 2,
    "large-vehicle": 3,
}


def load_labels(label_path):
    objects = read_dota_label(Path(label_path), class_to_id=CLASS_MAP)
    return [
        (obj.class_id, [coord for point in obj.polygon for coord in point])
        for obj in objects
    ]


def generate_tiles(img, tile_size=1024, overlap=200):
    windows = generate_tile_windows(img.shape[1], img.shape[0], tile_size=tile_size, overlap=overlap)
    return [
        (extract_padded_tile(img, window, tile_size), window.x, window.y, window.width, window.height)
        for window in windows
    ]


def adjust_labels(objects, x_offset, y_offset, tile_size, img_w, img_h, tile_w, tile_h):
    window = SourceObject, None
    tile_window = __import__("app.dataset_tiling", fromlist=["TileWindow"]).TileWindow(
        index=0,
        x=x_offset,
        y=y_offset,
        width=tile_w,
        height=tile_h,
    )
    source_objects = [
        SourceObject(
            class_id=cls_id,
            class_name=str(cls_id),
            polygon=tuple((coords[i] * img_w, coords[i + 1] * img_h) for i in range(0, 8, 2)),
        )
        for cls_id, coords in objects
    ]
    return build_tile_annotations(source_objects, tile_window, tile_size)


def process_split(split, tile_size=1024, overlap=200, empty_keep_ratio=0.10):
    img_dir = os.path.join(SRC_ROOT, "images", split)
    lbl_dir = os.path.join(SRC_ROOT, "labels", split)
    out_img_dir = os.path.join(DST_ROOT, "images", split)
    out_lbl_dir = os.path.join(DST_ROOT, "labels", split)

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    saved_tiles = 0
    saved_empty_tiles = 0
    img_files = [f for f in os.listdir(img_dir) if f.endswith(".png")]

    for img_name in tqdm(img_files, desc=f"Tiling {split}"):
        img_path = os.path.join(img_dir, img_name)
        lbl_path = os.path.join(lbl_dir, img_name.replace(".png", ".txt"))

        img = cv2.imread(img_path)
        if img is None:
            continue

        source_objects = read_dota_label(Path(lbl_path), class_to_id=CLASS_MAP)
        windows = generate_tile_windows(img.shape[1], img.shape[0], tile_size, overlap)

        for window in windows:
            new_objs = build_tile_annotations(source_objects, window, tile_size)
            if not new_objs and not deterministic_keep_empty_tile(
                img_name.replace(".png", ""),
                window.index,
                empty_keep_ratio,
                42,
            ):
                continue

            tile = extract_padded_tile(img, window, tile_size)
            tile_base = f"{img_name.replace('.png', '')}_{window.index:04d}"
            tile_img_name = f"{tile_base}.png"
            tile_lbl_name = f"{tile_base}.txt"

            cv2.imwrite(os.path.join(out_img_dir, tile_img_name), tile)
            with open(os.path.join(out_lbl_dir, tile_lbl_name), "w", encoding="utf-8") as handle:
                for cls_id, coords in new_objs:
                    handle.write(str(cls_id) + " " + " ".join(f"{coord:.6f}" for coord in coords) + "\n")

            saved_tiles += 1
            if not new_objs:
                saved_empty_tiles += 1

    print(f"\nSplit: {split}")
    print(f"Tiles sauvegardees: {saved_tiles}")
    print(f"Tiles vides conservees: {saved_empty_tiles}")


##Complete Pipeline tiling

In [ ]:
from pathlib import Path

from app.dataset_tiling import (
    SourceObject,
    build_tile_annotations,
    deterministic_keep_empty_tile,
    extract_padded_tile,
    generate_tile_windows,
    read_dota_label,
)

CLASS_MAP = {
    "plane": 0,
    "ship": 1,
    "small-vehicle": 2,
    "large-vehicle": 3,
}


def load_labels(label_path):
    objects = read_dota_label(Path(label_path), class_to_id=CLASS_MAP)
    return [
        (obj.class_id, [coord for point in obj.polygon for coord in point])
        for obj in objects
    ]


def generate_tiles(img, tile_size=1024, overlap=200):
    windows = generate_tile_windows(img.shape[1], img.shape[0], tile_size=tile_size, overlap=overlap)
    return [
        (extract_padded_tile(img, window, tile_size), window.x, window.y, window.width, window.height)
        for window in windows
    ]


def adjust_labels(objects, x_offset, y_offset, tile_size, img_w, img_h, tile_w, tile_h):
    window = SourceObject, None
    tile_window = __import__("app.dataset_tiling", fromlist=["TileWindow"]).TileWindow(
        index=0,
        x=x_offset,
        y=y_offset,
        width=tile_w,
        height=tile_h,
    )
    source_objects = [
        SourceObject(
            class_id=cls_id,
            class_name=str(cls_id),
            polygon=tuple((coords[i] * img_w, coords[i + 1] * img_h) for i in range(0, 8, 2)),
        )
        for cls_id, coords in objects
    ]
    return build_tile_annotations(source_objects, tile_window, tile_size)


def process_split(split, tile_size=1024, overlap=200, empty_keep_ratio=0.10):
    img_dir = os.path.join(SRC_ROOT, "images", split)
    lbl_dir = os.path.join(SRC_ROOT, "labels", split)
    out_img_dir = os.path.join(DST_ROOT, "images", split)
    out_lbl_dir = os.path.join(DST_ROOT, "labels", split)

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    saved_tiles = 0
    saved_empty_tiles = 0
    img_files = [f for f in os.listdir(img_dir) if f.endswith(".png")]

    for img_name in tqdm(img_files, desc=f"Tiling {split}"):
        img_path = os.path.join(img_dir, img_name)
        lbl_path = os.path.join(lbl_dir, img_name.replace(".png", ".txt"))

        img = cv2.imread(img_path)
        if img is None:
            continue

        source_objects = read_dota_label(Path(lbl_path), class_to_id=CLASS_MAP)
        windows = generate_tile_windows(img.shape[1], img.shape[0], tile_size, overlap)

        for window in windows:
            new_objs = build_tile_annotations(source_objects, window, tile_size)
            if not new_objs and not deterministic_keep_empty_tile(
                img_name.replace(".png", ""),
                window.index,
                empty_keep_ratio,
                42,
            ):
                continue

            tile = extract_padded_tile(img, window, tile_size)
            tile_base = f"{img_name.replace('.png', '')}_{window.index:04d}"
            tile_img_name = f"{tile_base}.png"
            tile_lbl_name = f"{tile_base}.txt"

            cv2.imwrite(os.path.join(out_img_dir, tile_img_name), tile)
            with open(os.path.join(out_lbl_dir, tile_lbl_name), "w", encoding="utf-8") as handle:
                for cls_id, coords in new_objs:
                    handle.write(str(cls_id) + " " + " ".join(f"{coord:.6f}" for coord in coords) + "\n")

            saved_tiles += 1
            if not new_objs:
                saved_empty_tiles += 1

    print(f"\nSplit: {split}")
    print(f"Tiles sauvegardees: {saved_tiles}")
    print(f"Tiles vides conservees: {saved_empty_tiles}")


In [ ]:
process_split("train")
process_split("val")

print("Tiling terminé 🚀")

In [ ]:
print("Train images:", len(os.listdir(f"{DST_ROOT}/images/train")))
print("Val images:", len(os.listdir(f"{DST_ROOT}/images/val")))

## Tiled Dataset Generation

To improve small object detection, we generate a new dataset using image tiling.

### Strategy

* Tile size: **1024 × 1024**
* Overlap: **200 pixels**
* Keep a small percentage of empty tiles to preserve background diversity

### Key Considerations

* Objects partially inside a tile are preserved if at least one corner is within the tile
* Bounding boxes are adjusted and re-normalized
* Padding is applied to maintain consistent tile size

### Result

* Original dataset → transformed into a larger tiled dataset
* Increased number of training samples
* Better representation of small objects


##  Final Training on Tiled Dataset

## Training on Tiled Dataset

We retrain the YOLO OBB model using the tiled dataset.

### Objective

* Improve detection of small objects
* Increase recall and localization accuracy

### Training Setup

* Pretrained model: YOLO11n-OBB
* Image size: 1024
* Data augmentation: enabled (flip, scale, mosaic, rotation)
* Early stopping applied

This training aims to evaluate the impact of tiling on model performance.


In [ ]:
yaml_content = f"""
path: {DST_ROOT}

train: images/train
val: images/val

names:
  0: plane
  1: ship
  2: small-vehicle
  3: large-vehicle
"""

with open("/content/data/dota_tiled.yaml", "w") as f:
    f.write(yaml_content)

print("YAML créé")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-obb.pt")

results = model.train(
    data="/content/data/dota_tiled.yaml",
    epochs=50,
    imgsz=1024,
    batch=4,
    device=0,
    workers=2,

    degrees=45,
    translate=0.1,
    scale=0.6,
    fliplr=0.5,
    flipud=0.5,
    mosaic=1.0,

    lr0=0.001,
    patience=15,

    project="/content/runs_obb",
    name="dota_tiled_from_pretrained"
)

In [ ]:
import pandas as pd

# Paths
run1_path = "/content/runs_obb/dota4_train/results.csv"
run2_path = "/content/runs_obb/dota_tiled_from_pretrained/results.csv"

# Load
run1 = pd.read_csv(run1_path)
run2 = pd.read_csv(run2_path)

# Last epoch (best training state)
run1_last = run1.iloc[-1]
run2_last = run2.iloc[-1]

print("Run 1 (Initial):")
print(run1_last)

print("\nRun 2 (Tiled):")
print(run2_last)

In [ ]:
comparison = pd.DataFrame({
    "Metric": ["mAP50", "mAP50-95", "Precision", "Recall"],
    "Initial": [
        run1_last["metrics/mAP50(B)"],
        run1_last["metrics/mAP50-95(B)"],
        run1_last["metrics/precision(B)"],
        run1_last["metrics/recall(B)"]
    ],
    "Tiled": [
        run2_last["metrics/mAP50(B)"],
        run2_last["metrics/mAP50-95(B)"],
        run2_last["metrics/precision(B)"],
        run2_last["metrics/recall(B)"]
    ]
})

comparison

In [ ]:
import matplotlib.pyplot as plt

comparison.set_index("Metric").plot(kind="bar")
plt.title("Model Comparison: Initial vs Tiled")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(run1["metrics/mAP50(B)"], label="Initial")
plt.plot(run2["metrics/mAP50(B)"], label="Tiled")

plt.title("mAP50 over Epochs")
plt.xlabel("Epoch")
plt.ylabel("mAP50")
plt.legend()
plt.grid()

plt.show()

## Training Dynamics Analysis

The training curves show that the tiled model consistently outperforms the baseline.

### Observations

* The tiled model achieves higher mAP50 across epochs
* Convergence is faster and more stable
* Less fluctuation in performance during training

### Interpretation

Tiling simplifies the learning problem by reducing scene complexity and increasing object visibility, allowing the model to learn more robust features.


## Final Comparison: Baseline vs Tiled Model

After applying the tiling strategy, the model shows a clear and consistent improvement over the baseline.

### Quantitative Comparison

* mAP50 improved from **0.757** → **0.883**
* mAP50-95 improved from **0.424** → **0.553**
* Precision improved from **0.886** → **0.901**
* Recall improved from **0.681** → **0.828**

### Small Object Detection

The most significant improvement is observed for the **small-vehicle** class:

* Baseline mAP50: **0.454**
* Tiled mAP50: **0.868**

This represents a major performance gain and confirms that tiling directly addresses the small object detection problem.

### Key Takeaway

Tiling increases the relative size of small objects and enhances local feature learning, leading to significantly improved detection performance in aerial imagery.


In [ ]:
import shutil

shutil.make_archive(
    base_name="/content/runs_obb_backup",
    format="zip",
    root_dir="/content",
    base_dir="runs_obb"
)

print("ZIP créé ✅")

### Deployment Consideration

Since the model was trained on tiled images, inference on full-resolution images requires applying the same tiling strategy at prediction time.

This involves:

* Splitting the input image into tiles
* Running inference on each tile
* Merging predictions back into the original image space

This step is essential to maintain performance consistency in real-world applications.


# Conclusion

This project demonstrates the importance of data-centric strategies in improving object detection performance on aerial imagery.

### Key Findings

* Oriented Bounding Box (OBB) detection is well-suited for aerial scenes with rotated objects.
* The baseline model performs well on large objects but struggles with small ones, particularly vehicles.
* Small object detection is strongly affected by scale and image resolution.

### Impact of Tiling

By applying a tiling strategy, we significantly improved model performance:

* mAP50 increased from **0.757** to **0.883**
* mAP50-95 increased from **0.424** to **0.553**
* Small-vehicle detection improved from **0.454** to **0.868**

This confirms that tiling is an effective solution for addressing scale-related limitations in high-resolution images.

### Key Takeaway

Improving data representation can be as impactful as changing the model itself.
In this case, a simple preprocessing strategy led to substantial performance gains.

### Next Steps

* Build an inference pipeline on full-resolution images using tiling + prediction merging
* Deploy the model as a FastAPI service
* Optimize inference speed and scalability
